# 08 · `manual_chunks.jsonl` — 절차 RAG 청크 (기능 1·5)

| | |
|---|---|
| **목적** | `rag_search_manual` 이 검색하는 청크의 분포·품질·인용 추적성 점검 |
| **input** | `data/extracted/manual_chunks.jsonl` (1,035청크 · `chunk_manual.py` 산출) |
| **output** | `data-analysis/data/processed/manual_chunks.parquet`<br>`data-analysis/reports/manual_chunks_decisions.json` |

> ⛔ **텍스트를 고치지 않는다.** 인용 페이지가 어긋나면 근거 추적이 무너진다(D26·D32).
> 재생성은 `chunk_manual.py` 로만.

> 🔍 **D1 경계**: 이 데이터는 **절차 서술**용이다. 에러코드 **정의** 조회는 `07` 의
> `error_codes.json` 이 exact match 로 담당한다 — 경계를 흐리는 코드는 금지(절대규칙 2).

In [ ]:
import sys
sys.path.insert(0, "..")
import json

import pandas as pd
from common import EXTRACTED, profile, save_decisions, save_processed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 원천 | LS 매뉴얼 PDF 3종 (iG5A 표준본 · iG5A 트러블슈팅 · S100) |
| 만든 것 | `data/chunk_manual.py` |
| 인용 기준 | **PDF 물리 페이지 무가공** (D26) |

In [ ]:
rows = [json.loads(line) for line in
        (EXTRACTED / "manual_chunks.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
df_raw = pd.DataFrame(rows)
print(f"원본 shape: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열")
print("컬럼:", list(df_raw.columns))
print(f"\n총 글자수: {df_raw['char_len'].sum():,}자")

## 2. 로드 + 기초 통계

In [ ]:
profile(df_raw)

In [ ]:
df_raw.head(3)[["chunk_id", "manual_id", "model", "page", "char_len"]]

### 2-1. 실제 청크 하나를 그대로 본다

In [ ]:
s = df_raw.iloc[400]
print("chunk_id :", s["chunk_id"])
print("manual_id:", s["manual_id"], "| model:", s["model"], "| page:", s["page"])
print("section  :", repr(s["section"][:70]))
print("char_len :", s["char_len"])
print("-" * 70)
print(s["text"][:400])

## 3. EDA

### 3-1. 매뉴얼·기종별 분포

In [ ]:
print(pd.crosstab(df_raw["manual_id"], df_raw["model"], margins=True).to_string())
print()
print("→ iG5A 386 (표준본 342 + 트러블슈팅 44) · S100 649.")
print("  S100 매뉴얼이 21.3MB 로 가장 두꺼워 청크도 가장 많다.")

### 3-2. `char_len` 분포 — 청킹 품질

In [ ]:
print(df_raw["char_len"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())
print()
print(f"평균 {df_raw['char_len'].mean():.0f}자 · 중앙 {df_raw['char_len'].median():.0f}자")
print(f"최소 {df_raw['char_len'].min()} · 최대 {df_raw['char_len'].max()}")
print()
buckets = pd.cut(df_raw["char_len"], [0, 100, 200, 400, 600, 800, 900])
print(buckets.value_counts().sort_index().to_string())
print()
print("→ 상한 900 · 하한 60 이 지켜졌다. 청킹기가 경계를 강제하고 있다.")

In [ ]:
print("기종별 char_len —")
print(df_raw.groupby("model")["char_len"].agg(["count", "mean", "median", "min", "max"]).round(1).to_string())

### 3-3. `section` 이 실제로 섹션 제목인가 — 필터로 쓸 수 있나

`section` 이 목차 제목이면 검색 필터로 쓸 수 있지만, **본문 첫 줄이 흘러들어와 있으면**
못 쓴다. **고유값이 346개나 되어 처음엔 본문이 새어 들어온 줄 알았는데, 실측은 반대였다.**

In [ ]:
sec = df_raw["section"]
길이 = sec.str.len()
print("section 길이 분포 —")
print(길이.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())
print()
번호형 = sec.str.match(r"^\d+(\.\d+)*[\s.]")
print(f"절 번호로 시작 ('1.' · '1.4' 등): {번호형.sum():,} / {len(df_raw):,}  ({번호형.mean() * 100:.1f}%)")
print(f"40자 이하                     : {(길이 <= 40).sum():,}  ({(길이 <= 40).mean() * 100:.1f}%)")
print(f"60자 초과 (본문이 흘러든 흔적)  : {(길이 > 60).sum()}건")

In [ ]:
print("section 샘플 —")
for v in sec.drop_duplicates().head(8):
    print(f"  {v!r}")
print()
g = df_raw.groupby(["model", "page"])["section"].nunique()
print(f"한 페이지에 section 이 하나뿐인 면: {(g == 1).sum()}/{len(g)}  ({(g == 1).mean() * 100:.1f}%)")

> ✅ **`section` 은 정상적인 섹션 제목이다. 필터로 쓸 수 있다.**
> 평균 **16.9자** · 최대 42자 · **90.8%가 절 번호로 시작** · 60자 초과 **0건** ·
> 페이지의 90%가 단일 섹션. 고유값 346개는 **매뉴얼 3종의 실제 절 수**이지 본문 유출이 아니다.
>
> ⚠ 예외 하나 — 표지·목차 페이지의 `section` 은 절 번호가 없다
> (`'LS ELECTRIC에서는 저희 제품을…'` · `'목 차'`). 필터를 걸 때 이 구간은 빠진다.
>
> 📌 **초안에서는 이 컬럼을 "본문 앞부분이라 못 쓴다"고 판정했다가 뒤집었다.**
> `text.startswith(section)` 이 20.7% 라는 수치를 오해한 것이었는데,
> 그건 **각 절의 첫 청크에서 제목이 본문 맨 앞에 오는 게 당연**해서 나오는 값이다.

### 3-4. 🚨 페이지 커버리지 — 인용 앵커가 비어 있는 구간은 없나

In [ ]:
for m, g in df_raw.groupby("model"):
    pages = sorted(g["page"].unique())
    구멍 = [p for p in range(min(pages), max(pages) + 1) if p not in set(pages)]
    print(f"{m:6} 페이지 {min(pages)}~{max(pages)} · 청크 있는 페이지 {len(pages)}개 · "
          f"빈 페이지 {len(구멍)}개")
    if 구멍:
        print(f"       빈 구간 예: {구멍[:12]}{' …' if len(구멍) > 12 else ''}")
print()
print("→ 빈 페이지는 표/그림만 있는 면이다. 텍스트가 없으면 청크도 없는 게 정상이다.")

### 3-5. 🚨 `07` 과의 교차 검증 — 에러코드가 인용한 페이지에 청크가 있는가

**이 노트북에서 가장 중요한 검사다.** `error_codes.json` 의 `manual_page` 가 가리키는 곳에
청크가 없으면, 진단은 되는데 **근거를 보여줄 수 없다.**

In [ ]:
ec = pd.DataFrame(json.loads(
    (EXTRACTED / "error_codes.json").read_text(encoding="utf-8"))["entries"])
있는페이지 = set(zip(df_raw["model"], df_raw["page"], strict=True))
ec["청크있음"] = [(m, p) in 있는페이지 for m, p in
                zip(ec["model"], ec["manual_page"], strict=True)]

print(f"에러코드 {len(ec)}건 중 인용 페이지에 청크가 있는 건: {ec['청크있음'].sum()}건")
print(f"청크 없는 건: {(~ec['청크있음']).sum()}건")
print()
print(pd.crosstab(ec["model"], ec["청크있음"]).to_string())
if (~ec["청크있음"]).any():
    print("\n⚠ 근거를 못 보여주는 코드 —")
    print(ec[~ec["청크있음"]][["model", "code", "manual_page"]].to_string(index=False))
else:
    print("\n✅ 전건 커버. 모든 에러코드가 인용 페이지의 청크로 근거를 보여줄 수 있다.")

### 3-6. 인용 페이지의 청크 밀도

In [ ]:
인용면 = ec[["model", "manual_page"]].drop_duplicates()
밀도 = (df_raw.groupby(["model", "page"]).size().rename("청크수").reset_index()
       .merge(인용면, left_on=["model", "page"], right_on=["model", "manual_page"]))
print("에러코드가 인용하는 페이지의 청크 수 —")
print(밀도[["model", "page", "청크수"]].sort_values(["model", "page"]).to_string(index=False))
print()
print(f"인용 대상 페이지 {len(밀도)}면 · 평균 {밀도['청크수'].mean():.1f}청크/면")

## 4. 전처리

In [ ]:
df = df_raw.copy()

# chunk_id 를 분해 — 'ig5a-manual-p0001-00' = manual_id + page + 면내순번
분해 = df["chunk_id"].str.extract(r"^(?P<_mid>.+)-p(?P<_page>\d{4})-(?P<면내순번>\d+)$")
df["면내순번"] = pd.to_numeric(분해["면내순번"], errors="coerce").astype("Int64")

# chunk_id 의 page 가 page 컬럼과 일치하는지 — 어긋나면 인용이 깨진다
일치 = pd.to_numeric(분해["_page"], errors="coerce") == df["page"]
print(f"chunk_id 의 페이지 == page 컬럼: {일치.sum():,} / {len(df):,}  "
      f"{'✅ 전건 일치' if 일치.all() else '❌ 불일치 있음'}")

df["char_len_실측"] = df["text"].str.len()
어긋남 = (df["char_len"] != df["char_len_실측"]).sum()
print(f"char_len 필드 == len(text): {len(df) - 어긋남:,} / {len(df):,}  "
      f"{'✅' if 어긋남 == 0 else f'❌ {어긋남}건 어긋남'}")

df["에러코드인용면"] = [
    (m, p) in set(zip(ec["model"], ec["manual_page"], strict=True))
    for m, p in zip(df["model"], df["page"], strict=True)
]
print(f"\n정제 후: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"에러코드가 인용하는 면의 청크: {df['에러코드인용면'].sum()}개")

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "chunk_id", "used": True,
     "reason": "청크 고유 식별자. manual_id+page+면내순번 구조라 페이지 정합성 자가 검증에 쓴다"},
    {"column": "manual_id", "used": True, "reason": "manifest.json 의 매뉴얼 키 — 버전·해시 추적의 연결점(D19)"},
    {"column": "model", "used": True, "reason": "🚨 검색 스코프. enum 2종(D6·D13). 기종을 안 가르면 다른 기계의 절차가 나온다"},
    {"column": "page", "used": True, "reason": "🚨 인용 앵커. PDF 물리 페이지 무가공(D26). 표시 변환만 오프셋(D32)"},
    {"column": "text", "used": True, "reason": "검색 본문. RAG 가 실제로 읽는 것"},
    {"column": "char_len", "used": True, "reason": "청킹 품질 지표. len(text) 와 일치 검증에 사용"},
    {"column": "section", "used": True,
     "reason": "정상적인 섹션 제목이다 — 평균 16.9자 · 90.8%가 절 번호로 시작 · 60자 초과 0건. "
               "검색 범위를 절 단위로 좁히는 필터로 쓸 수 있다. "
               "⚠ 단 표지·목차 페이지는 절 번호가 없어 필터에서 빠진다"},
]
assert len(column_decisions) == df_raw.shape[1], \
    f"결정 {len(column_decisions)} vs 원본 컬럼 {df_raw.shape[1]}"
print(f"컬럼 결정 {len(column_decisions)}건 — 원본 컬럼 수와 일치")

In [ ]:
save_decisions(
    "manual_chunks",
    column_decisions,
    source="LS 매뉴얼 PDF 3종",
    generator="data/chunk_manual.py",
    row_count_raw=int(df_raw.shape[0]),
    col_count_raw=int(df_raw.shape[1]),
    total_chars=int(df_raw["char_len"].sum()),
    avg_chars=round(float(df_raw["char_len"].mean()), 1),
    by_manual=df_raw["manual_id"].value_counts().to_dict(),
    error_code_page_coverage={
        "total": int(len(ec)), "covered": int(ec["청크있음"].sum()),
        "missing": int((~ec["청크있음"]).sum()),
    },
    generated_columns=["면내순번", "char_len_실측", "에러코드인용면"],
    caveats=[
        "⛔ 텍스트·page 를 고치면 인용 추적이 무너진다(D26·D32) — 재생성은 chunk_manual.py 로만",
        "section 은 쓸 수 있는 섹션 제목이다 (평균 16.9자 · 90.8% 절 번호). "
        "단 표지·목차 페이지는 절 번호가 없어 필터에서 빠진다",
        "빈 페이지는 표·그림만 있는 면이다. 텍스트가 없으면 청크가 없는 게 정상",
        "D1 경계: 이 데이터는 절차 서술용이다. 코드 정의 조회는 error_codes.json 의 exact match",
        "✅ 에러코드 65건 전부 인용 페이지에 청크가 있다 — 근거를 못 보여주는 코드 0건",
    ],
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("manual_chunks", df)